In [1]:
#
#this reads zonal data for various variables and generates one file that will be used in development of forecast model
#
import xarray as xr
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
import geopandas as gpd
import os, sys, glob, json, itertools

import warnings
warnings.filterwarnings("ignore")

In [18]:
#trigger model will be calcualated for Jan-April of this year
triggeryear=2026
triggermonth=4

monthnames=["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

#these are files prepared in a GIS
studyarea = gpd.read_file("../data/gis/study_area.shp")
studyregions=studyarea["NAME_1"]

In [19]:
#
#spi data
#
#this has to be changed by hand to the file with the most recent data
spifile="./data/zonal_data/spi6_mon_era5_198001-202603.csv"

#reading data
spi6_era5=pd.read_csv(spifile, parse_dates=True, index_col=0)

#aligning dates - spi is dated on the last month of the 6month period for which data are available
#here we are shifting it by one month - to the month on which the trigger is calculated.
# so trigger calculated in April will use SPI data from March
spi6_era5.index=spi6_era5.index.normalize()+pd.offsets.MonthBegin(1)

#we want to create a pandas dataframe with two-level column index, one for variable (spi6_era5) and one for provinces

#creating dictionary with spi data
spidata=spi6_era5.copy()

mindex=list(zip(["spi6_era5"]*spidata.shape[1], spidata.columns))
mindex=pd.MultiIndex.from_tuples(mindex,names=["variable", "province"])
spidata.columns=mindex

print(spidata)
#spi6_era5

variable   spi6_era5                                                    \
province     Badghis     Balkh    Faryab     Hirat   Jawzjan  Samangan   
time                                                                     
1980-02-01       NaN       NaN       NaN       NaN       NaN       NaN   
1980-03-01       NaN       NaN       NaN       NaN       NaN       NaN   
1980-04-01       NaN       NaN       NaN       NaN       NaN       NaN   
1980-05-01       NaN       NaN       NaN       NaN       NaN       NaN   
1980-06-01       NaN       NaN       NaN       NaN       NaN       NaN   
...              ...       ...       ...       ...       ...       ...   
2025-12-01 -2.911502 -2.842409 -2.942128 -2.426775 -3.076269 -2.717657   
2026-01-01 -0.450058 -2.007608 -1.005981  0.873144 -1.026344 -3.090000   
2026-02-01 -0.866197 -1.779391 -1.186017  0.265827 -0.945448 -2.914017   
2026-03-01 -1.224149 -2.571290 -1.684746  0.001451 -1.411294 -3.090000   
2026-04-01 -0.566274 -1.426512 -0.8879

In [20]:
#
#loading ENSO

enso=pd.read_csv("./data/ENSO/nino34_psl.csv", parse_dates=True, index_col=0)
sel=~np.isnan(enso)
enso=enso[sel.values]

#aligning enso time with trigger model time
enso.index=enso.index+pd.offsets.MonthBegin()

#we want to create a pandas dataframe with two-level column index, one for variable (spi6_era5) and one for provinces

ensodata=enso.copy()

ensodata=pd.DataFrame(np.tile(enso.values, len(studyregions)), index=enso.index, columns=studyregions)
mindex=list(zip(["enso"]*ensodata.shape[1], ensodata.columns))
mindex=pd.MultiIndex.from_tuples(mindex,names=["variable", "province"])
ensodata.columns=mindex
print(ensodata)

variable      enso                                              
province   Badghis Balkh Faryab Hirat Jawzjan Samangan Sar-e-Pul
1950-02-01   -1.99 -1.99  -1.99 -1.99   -1.99    -1.99     -1.99
1950-03-01   -1.69 -1.69  -1.69 -1.69   -1.69    -1.69     -1.69
1950-04-01   -1.42 -1.42  -1.42 -1.42   -1.42    -1.42     -1.42
1950-05-01   -1.54 -1.54  -1.54 -1.54   -1.54    -1.54     -1.54
1950-06-01   -1.75 -1.75  -1.75 -1.75   -1.75    -1.75     -1.75
...            ...   ...    ...   ...     ...      ...       ...
2025-12-01   -0.70 -0.70  -0.70 -0.70   -0.70    -0.70     -0.70
2026-01-01   -0.67 -0.67  -0.67 -0.67   -0.67    -0.67     -0.67
2026-02-01   -0.58 -0.58  -0.58 -0.58   -0.58    -0.58     -0.58
2026-03-01   -0.27 -0.27  -0.27 -0.27   -0.27    -0.27     -0.27
2026-04-01   -0.06 -0.06  -0.06 -0.06   -0.06    -0.06     -0.06

[915 rows x 7 columns]


In [21]:
#
#reading forecast data
#
#this has to be edited by hand to pick up file with correct date
fcstfile="./data/zonal_data/droughtprob_mon_seas51_198101-202604.nc"


dset=xr.open_dataset(fcstfile)
fcstprob=dset["probability"]

fcstprob["time"]=pd.to_datetime(fcstprob.time).normalize()

#aligning forecast target time with trigger model time
#time here is the last month of 3-month forecast. That can be forecasted with 2 month lead time - in pythonic notation!- 
# so for forecast of May (MAM) at 2 month lead time it will be issued in March

tempdata=[]
for lt in fcstprob.leadtime:
    print(lt.data)
    temp=fcstprob.sel(leadtime=lt).to_pandas().copy()
    #going back to the time when forecast is issued
    temp.index=temp.index-pd.offsets.MonthBegin(lt)
    mindex=list(zip(["pr-fcst_{}".format(lt.values)]*temp.shape[1], temp.columns.values))
    mindex=pd.MultiIndex.from_tuples(mindex,names=["variable", "province"])
    temp.columns=mindex
    tempdata.append(temp)
fcstdata=pd.concat(tempdata, axis=1)
print(fcstdata)

2
3
4
5
variable    pr-fcst_2                                                         \
province      Badghis      Balkh     Faryab      Hirat    Jawzjan   Samangan   
time                                                                           
1980-10-01        NaN        NaN        NaN        NaN        NaN        NaN   
1980-11-01        NaN        NaN        NaN        NaN        NaN        NaN   
1980-12-01        NaN        NaN        NaN        NaN        NaN        NaN   
1981-01-01  24.000000  28.000000  20.000000  28.000000  20.000000  28.000000   
1981-02-01  28.000000  40.000000  36.000000  36.000000  36.000000  44.000000   
...               ...        ...        ...        ...        ...        ...   
2026-03-01  11.764706   0.000000   9.803922  15.686275   9.803922   1.960784   
2026-04-01   0.000000   9.803922   3.921569   0.000000   7.843137   5.882353   
2026-05-01        NaN        NaN        NaN        NaN        NaN        NaN   
2026-06-01        NaN        NaN

In [22]:
#
#reading vegetation indices and soil moisture
#
lstfile="./data/zonal_data/lst_modis_200002-202603.csv"

lstdata=pd.read_csv(lstfile, parse_dates=True, index_col=0)
mindex=list(zip(["LST"]*lstdata.shape[1], lstdata.columns))
mindex=pd.MultiIndex.from_tuples(mindex,names=["variable", "province"])
lstdata.columns=mindex

lstdata.index=lstdata.index+pd.offsets.MonthBegin(1)

print(lstdata)

variable           LST                                                  \
province       Badghis       Balkh      Faryab       Hirat     Jawzjan   
2000-03-01  283.314375  292.433708  284.307574  293.423517  293.662207   
2000-04-01  293.141747  297.195480  293.249700  298.588749  299.857226   
2000-05-01  309.355425  311.006564  310.080581  311.871078  315.248966   
2000-06-01  318.942708  318.922307  319.608154  318.822528  322.549971   
2000-07-01  319.965400  321.985507  321.903325  318.065765  325.736597   
...                ...         ...         ...         ...         ...   
2025-12-01  289.343432  289.857257  289.281275  291.908213  290.772075   
2026-01-01  277.484074  280.785657  276.801054  281.067727  279.754801   
2026-02-01  274.043901  277.780114  275.314363  277.353854  279.151866   
2026-03-01  282.916000  286.366171  284.181054  285.672516  288.046289   
2026-04-01  286.261926  290.834000  287.699069  289.694032  292.134340   

variable                            


In [23]:
#temporal subsetting and concatenating all data into a single dataframe
fy,ly="2000","2026"
alldata=pd.concat([ensodata[fy:ly], 
                   lstdata[fy:ly], fcstdata[fy:ly], spidata[fy:ly]], axis=1)

outputfile="./data/zonal_data/alldata_{}{}.csv".format(triggeryear, str(triggermonth).zfill(2))
alldata.to_csv(outputfile)
print("written ",outputfile)

written  ./data/zonal_data/alldata_202604.csv
